# Gold | Fato-Chamados


In [0]:
%run ./utils

# Imports

In [0]:
import pyspark.sql.functions as F

# Load

In [0]:
dbutils.widgets.text("gold_table_name", "fato_chamados")
gold_table_name = dbutils.widgets.get("gold_table_name")

In [0]:
df_chamados = (
    spark.table(f"{silver_schema}.tb_chamados")
        .select("id_chamado", "id_cliente", "canal", "motivo", "id_atendente", "resolvido")
)

print(f"Total de linhas de chamados: {df_chamados.count()}")

In [0]:
df_horarios = (
    spark.table(f"{silver_schema}.tb_chamados_hora")
        .select(
            "id_chamado",
            "hora_abertura_chamado",
            "hora_inicio_atendimento",
            "hora_finalizacao_atendimento",
            "tempo_espera_segundos",
            "tempo_atendimento_segundos"
        )
)

print(f"Total de linhas de horários: {df_horarios.count()}")

In [0]:
# Custo por chamado (no máximo um custo por chamado na origem; a soma evita duplicar linhas)
df_custos = (
    spark.table(f"{silver_schema}.tb_custos")
        .groupBy("id_chamado")
        .agg(F.sum("custo").cast("decimal(18,2)").alias("custo_chamado"))
)

# Nota por chamado (as pesquisas repetidas do mesmo chamado trazem a mesma nota)
df_notas = (
    spark.table(f"{silver_schema}.tb_pesquisas_de_satisfacao")
        .groupBy("id_chamado")
        .agg(F.max("nota_atendimento").alias("nota_atendimento"))
)

In [0]:
# Chaves substitutas das dimensões
df_dim_cliente = spark.table(f"{gold_schema}.dim_cliente").select("cliente_sk", "id_cliente")
df_dim_atendente = spark.table(f"{gold_schema}.dim_atendente").select("atendentes_sk", "id_atendente")
df_dim_canal = spark.table(f"{gold_schema}.dim_canal").select("canal_sk", F.col("nome_canal").alias("canal"))
df_dim_motivo = spark.table(f"{gold_schema}.dim_motivo").select("motivo_sk", F.col("nome_motivo").alias("motivo"))
df_dim_data = spark.table(f"{gold_schema}.dim_data").select("data_sk", "data")
df_dim_horario = spark.table(f"{gold_schema}.dim_horario").select("horario_sk", "horario")

# Tratamento



In [0]:
# Junta chamados com horários, custos e notas (por id_chamado)
df_medidas = (
    df_chamados
    .join(df_horarios, "id_chamado", "left")
    .join(df_custos, "id_chamado", "left")
    .join(df_notas, "id_chamado", "left")
)

In [0]:
# Troca os atributos de negócio pelas chaves substitutas
df_dims = (
    df_medidas
    .join(df_dim_cliente, "id_cliente", "left")
    .join(df_dim_atendente, "id_atendente", "left")
    .join(df_dim_canal, "canal", "left")
    .join(df_dim_motivo, "motivo", "left")
    .withColumn("data", F.date_format("hora_abertura_chamado", "yyyyMMdd").cast("int"))
    .join(df_dim_data, "data", "left")
)

In [0]:
# Busca a chave de horário (formato HHmmss) de uma coluna de timestamp
def adicionar_horario_sk(df, coluna, nome_sk):
    return (
        df.withColumn("horario", F.date_format(coluna, "HHmmss").cast("int"))
          .join(df_dim_horario.withColumnRenamed("horario_sk", nome_sk), "horario", "left")
          .drop("horario")
    )

In [0]:
df_horario_sk = adicionar_horario_sk(df_dims, "hora_abertura_chamado", "horario_abertura_sk")
df_horario_sk = adicionar_horario_sk(df_horario_sk, "hora_inicio_atendimento", "horario_inicio_sk")
df_horario_sk = adicionar_horario_sk(df_horario_sk, "hora_finalizacao_atendimento", "horario_final_sk")

In [0]:
# Define as colunas finais, os tipos e a ordem
df_fato = df_horario_sk.select(
    "id_chamado",
    "cliente_sk",
    "motivo_sk",
    "canal_sk",
    "data_sk",
    "horario_abertura_sk",
    "horario_inicio_sk",
    "horario_final_sk",
    "atendentes_sk",
    "resolvido",
    F.col("tempo_espera_segundos").cast("int").alias("tempo_espera"),
    F.col("tempo_atendimento_segundos").cast("int").alias("tempo_atendimento"),
    "custo_chamado",
    "nota_atendimento"
).orderBy("id_chamado")

## Conferências

In [0]:
df_tempos_origem = spark.table(f"{silver_schema}.tb_chamados").select("id_chamado", "tempo_espera")

divergentes = (
    df_fato.join(df_tempos_origem.withColumnRenamed("tempo_espera", "tempo_espera_origem"), "id_chamado")
        .filter(F.col("tempo_espera_origem").isNotNull() & (F.abs(F.col("tempo_espera_origem") - F.col("tempo_espera")) > 1))
        .count()
)

print(f"Chamados com tempo de espera divergente entre as duas fontes: {divergentes}")
print(f"Chamados sem cliente na dimensão: {df_fato.filter(F.col('cliente_sk').isNull()).count()}")
print(f"Chamados sem custo: {df_fato.filter(F.col('custo_chamado').isNull()).count()}")
print(f"Chamados sem nota: {df_fato.filter(F.col('nota_atendimento').isNull()).count()}")

In [0]:
dq_check_unique("fato_chamados", "id_chamado único", df_fato, ["id_chamado"])

In [0]:
# Nenhum chamado da silver pode ter ficado de fora da fato
df_perdidos = df_chamados.join(df_fato.select("id_chamado"), "id_chamado", "left_anti")

dq_check(
    "fato_chamados",
    "todo chamado da silver está na fato",
    df_perdidos,
    F.lit(False)
)

In [0]:
dq_check(
    "fato_chamados",
    "chaves obrigatórias preenchidas",
    df_fato,
    F.col("motivo_sk").isNotNull()
    & F.col("canal_sk").isNotNull()
    & F.col("data_sk").isNotNull()
    & F.col("horario_abertura_sk").isNotNull()
    & F.col("horario_final_sk").isNotNull()
)

In [0]:
dq_check(
    "fato_chamados",
    "tempos não podem ser negativos",
    df_fato,
    (F.col(https://expresso.ac/pt/daily"tempo_espera").isNull() | (F.col("tempo_espera") >= 0))
    & (F.col("tempo_atendimento").isNull() | (F.col("tempo_atendimento") >= 0))
)

In [0]:
dq_check(
    "fato_chamados",
    "nota_atendimento entre 1 e 5",
    df_fato,
    F.col("nota_atendimento").isNull() | F.col("nota_atendimento").between(1, 5)
)

# Salvamento

In [0]:
save_df(df_fato, gold_table_name)

## Validação final

In [0]:
df_gold = spark.table(f"{gold_schema}.{gold_table_name}")

print(f"Linhas na gold: {df_gold.count()}")
display(df_gold.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_gold.columns]))
display(df_gold.groupBy("canal_sk").count().orderBy("canal_sk"))